<a href="https://colab.research.google.com/github/simone-agliano-dev/chatbot/blob/main/week1_setup_and_first_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic Customer Data Assistant — Week 1
## Load the CRM data into DuckDB + build the first agent tool

**Goal of this notebook:** get the data loaded, get one tool working end-to-end, and confirm the churn signal is queryable. No LLM yet — we build the foundation the agent will stand on.

**Prerequisites:** unzip `crm_dataset.zip` so there's a `data/` folder next to this notebook, containing the 5 CSVs and a `notes/` folder.

```
your-project/
├── week1_setup_and_first_tool.ipynb   <- this file
└── data/
    ├── accounts.csv
    ├── contacts.csv
    ├── opportunities.csv
    ├── activities.csv
    ├── support_tickets.csv
    └── notes/  (150 .txt files)
```

In [ ]:
# Unzip the archive: creates the data/ folder next to the notebook
import zipfile, os

with zipfile.ZipFile("crm_dataset.zip", "r") as z:
    z.extractall(".")

# Verify everything is there
print("Files/folders present:", os.listdir("."))
print("Inside data/:", os.listdir("data"))
print("Number of notes:", len(os.listdir("data/notes")))

Files/folders present: ['.config', 'drive', 'crm_dataset.zip', 'data', 'sample_data']
Inside data/: ['notes', 'activities.csv', 'opportunities.csv', 'support_tickets.csv', 'contacts.csv', 'accounts.csv']
Number of notes: 150


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# If running in Colab or a fresh env, uncomment:
#!pip install duckdb pandas

import duckdb
import pandas as pd
from pathlib import Path

DATA = Path("data")
assert DATA.exists(), "No data/ folder found. Unzip crm_dataset.zip next to this notebook."
print("duckdb", duckdb.__version__)

duckdb 1.3.2


## 1. Load the CSVs into DuckDB

DuckDB reads CSVs directly. We persist to a file (`crm.duckdb`) so the agent can reconnect later without re-loading.

In [ ]:
con = duckdb.connect("crm.duckdb")

TABLES = ["accounts", "contacts", "opportunities", "activities", "support_tickets"]
for t in TABLES:
    con.execute(
        f"CREATE OR REPLACE TABLE {t} AS "
        f"SELECT * FROM read_csv_auto('{DATA}/{t}.csv', header=true)"
    )

for t in TABLES:
    n = con.execute(f"SELECT count(*) FROM {t}").fetchone()[0]
    print(f"{t:16s} {n:5d} rows")

accounts           150 rows
contacts           384 rows
opportunities      287 rows
activities        1166 rows
support_tickets    329 rows


## 2. Inspect the schema

The agent will need to know the schema to write correct SQL. Let's capture it in a form we can later hand to the LLM as context.

In [ ]:
def get_schema_text(con):
    """Return a compact text description of all tables and columns."""
    lines = []
    for t in TABLES:
        cols = con.execute(f"PRAGMA table_info('{t}')").fetchall()
        col_str = ", ".join(f"{c[1]} {c[2]}" for c in cols)
        lines.append(f"{t}({col_str})")
    return "\n".join(lines)

schema_text = get_schema_text(con)
print(schema_text)

accounts(account_id VARCHAR, account_name VARCHAR, industry VARCHAR, tier VARCHAR, region VARCHAR, arr_usd BIGINT, customer_since DATE, health_status VARCHAR)
contacts(contact_id VARCHAR, account_id VARCHAR, name VARCHAR, title VARCHAR, email VARCHAR, is_champion BOOLEAN)
opportunities(opportunity_id VARCHAR, account_id VARCHAR, name VARCHAR, stage VARCHAR, amount_usd BIGINT, close_date DATE, is_won BOOLEAN)
activities(activity_id VARCHAR, account_id VARCHAR, type VARCHAR, activity_date DATE, notes_short VARCHAR)
support_tickets(ticket_id VARCHAR, account_id VARCHAR, priority VARCHAR, status VARCHAR, opened_date DATE, subject VARCHAR)


## 3. The first tool: `query_structured_data`

This is the function the agent will call. It takes a SQL string, runs it read-only, and returns rows as a list of dicts.

**Two things that matter for an FDE-grade tool (and make good interview talking points):**
1. **Safety guardrail** — we reject anything that isn't a `SELECT`, so the agent can't (accidentally or via prompt injection) mutate data.
2. **Bounded output** — we cap rows returned, so a runaway query doesn't blow up the LLM context.

In [ ]:
class QueryError(Exception):
    pass

def query_structured_data(sql: str, max_rows: int = 100) -> list[dict]:
    """Run a read-only SQL SELECT against the CRM database.

    This is the tool the agent will call. Returns rows as list of dicts.
    Raises QueryError on anything that isn't a safe SELECT.
    """
    cleaned = sql.strip().rstrip(";").strip()
    lowered = cleaned.lower()

    # Guardrail 1: only SELECT / WITH statements
    if not (lowered.startswith("select") or lowered.startswith("with")):
        raise QueryError("Only SELECT queries are allowed.")

    # Guardrail 2: block obvious mutation keywords defensively
    forbidden = ["insert ", "update ", "delete ", "drop ", "alter ", "create ", "attach "]
    if any(k in lowered for k in forbidden):
        raise QueryError("Query contains a forbidden keyword.")

    df = con.execute(cleaned).fetchdf()
    if len(df) > max_rows:
        df = df.head(max_rows)
    return df.to_dict(orient="records")

In [ ]:
# Try it: the core churn question, answered with real data
rows = query_structured_data("""
    SELECT account_name, arr_usd
    FROM accounts
    WHERE tier = 'Enterprise' AND health_status = 'At Risk'
    ORDER BY arr_usd DESC
    LIMIT 5
""")
pd.DataFrame(rows)

,account_name,arr_usd
0,Gonzalez Group,1151092
1,"Powell, Monroe and Caldwell",983028
2,Ray and Sons,965470
3,Jensen Ltd,938927
4,Carr-Mack,861759


In [ ]:
# Prove the planted signal is real: urgent open tickets cluster on at-risk accounts
rows = query_structured_data("""
    SELECT a.health_status, count(*) AS open_urgent_tickets
    FROM support_tickets t
    JOIN accounts a USING(account_id)
    WHERE t.priority IN ('High','Critical')
      AND t.status IN ('Open','In Progress')
    GROUP BY a.health_status
    ORDER BY open_urgent_tickets DESC
""")
pd.DataFrame(rows)

,health_status,open_urgent_tickets
0,At Risk,98
1,Healthy,5
2,Neutral,2


In [ ]:
# Confirm the guardrail works — this SHOULD raise
try:
    query_structured_data("DELETE FROM accounts")
    print("WARNING: guardrail failed!")
except QueryError as e:
    print("Guardrail OK ->", e)

Guardrail OK -> Only SELECT queries are allowed.


## 4. Where this is going (Week 2 preview)

You now have:
- data loaded in DuckDB,
- a schema description ready to hand to an LLM,
- one working, guardrailed tool.

**Next week** we wire this to Bedrock: the LLM receives the schema + the question, *decides* to call `query_structured_data`, writes the SQL itself, and we feed the results back for a final natural-language answer. Then we add the RAG tool over the `notes/` folder and the `calculate_churn_risk` domain tool, so the agent has to *choose* between tools — the heart of the agentic pattern.

**Interview note to self:** start keeping a debugging journal now. Every time the agent picks the wrong tool or writes bad SQL, write down what you changed to fix it. Those stories are what separate 'did a tutorial' from 'deployed an agent'.

In [ ]:
from google.colab import userdata
!pip install anthropic --quiet
import anthropic

api_key = userdata.get("ANTROPIC_KEY")
client = anthropic.Anthropic(api_key=api_key)

print("Client ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.4 MB/s eta 0:00:00
Client ready.


In [ ]:
# The most basic interaction: send a message, get a text reply. No tools yet.
response = client.messages.create(
    model="claude-sonnet-4-5",          # the model doing the reasoning
    max_tokens=300,                     # cap on how long the reply can be
    messages=[
        {"role": "user", "content": "In one sentence, what is a CRM?"}
    ],
)

# The reply comes back as a list of content blocks; here it's a single text block
print(response.content[0].text)

A CRM (Customer Relationship Management) is a system that helps businesses manage, track, and improve their interactions and relationships with customers and potential customers.


In [ ]:
# We describe the tool to the model as structured text.
# This description is the "prompt" that guides whether and how the model uses it.
TOOLS = [
    {
        "name": "query_structured_data",
        "description": (
            "Run a read-only SQL SELECT query against the CRM database. "
            "Use this for ANY question about numbers, counts, filters, or "
            "aggregations over structured data: accounts, ARR, tiers, regions, "
            "opportunities, support tickets, activities. "
            "Do NOT use this to search the free-text account notes. "
            "String comparisons are case-sensitive, so use the EXACT values below.\n"
            "The database is DuckDB. Available tables and columns:\n"
            "accounts(account_id, account_name, industry, tier, region, "
            "arr_usd, customer_since, health_status)\n"
            "  - tier is one of: 'Enterprise', 'Mid-Market', 'SMB'\n"
            "  - region is one of: 'EMEA', 'AMER', 'APAC'\n"
            "  - health_status is one of: 'Healthy', 'Neutral', 'At Risk'\n"
            "contacts(contact_id, account_id, name, title, email, is_champion)\n"
            "opportunities(opportunity_id, account_id, name, stage, amount_usd, "
            "close_date, is_won)\n"
            "  - stage is one of: 'Prospecting', 'Qualification', 'Proposal', "
            "'Negotiation', 'Closed Won', 'Closed Lost'\n"
            "activities(activity_id, account_id, type, activity_date, notes_short)\n"
            "  - type is one of: 'Email', 'Call', 'Meeting', 'Demo', 'QBR'\n"
            "support_tickets(ticket_id, account_id, priority, status, "
            "opened_date, subject)\n"
            "  - priority is one of: 'Low', 'Medium', 'High', 'Critical'\n"
            "  - status is one of: 'Open', 'In Progress', 'Resolved', 'Closed'"
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "sql": {
                    "type": "string",
                    "description": "A single read-only SQL SELECT statement.",
                }
            },
            "required": ["sql"],
        },
    }
]

In [ ]:
# Same messages.create as Step 1, but now we pass the tools.
question = "How many Enterprise accounts are marked as at risk?"

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=500,
    tools=TOOLS,                 # <-- the new part: the model now knows the tool exists
    messages=[
        {"role": "user", "content": question}
    ],
)

# Inspect WHY the model stopped and WHAT it produced
print("stop_reason:", response.stop_reason)
print()
for block in response.content:
    print("block type:", block.type)
    if block.type == "text":
        print("  text:", block.text)
    elif block.type == "tool_use":
        print("  tool name:", block.name)
        print("  tool input:", block.input)

NameError: name 'TOOLS' is not defined

In [ ]:
import json

question = "How many Enterprise accounts are marked as at risk?"

# --- First call: the model decides to use the tool (same as Step 2) ---
messages = [{"role": "user", "content": question}]

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=500,
    tools=TOOLS,
    messages=messages,
)

# --- Move 1: find the tool_use block in the model's reply ---
tool_use = next(b for b in response.content if b.type == "tool_use")
print("Model wants to run SQL:")
print("  ", tool_use.input["sql"])

# --- Move 2: actually execute the SQL ---
result = query_structured_data(tool_use.input["sql"])
print("Query result:", result)

# --- Move 3: rebuild the conversation, adding the model's turn + our tool_result ---
messages.append({"role": "assistant", "content": response.content})  # the model's request
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use.id,          # links the result to the request
            "content": json.dumps(result),       # the data, as a string
        }
    ],
})

# --- Move 4: call the model again, now with the result available ---
final = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=500,
    tools=TOOLS,
    messages=messages,
)

print("\nFinal answer:")
print(final.content[0].text)

NameError: name 'TOOLS' is not defined

In [ ]:
import json

def run_agent(question, max_turns=10, verbose=True):
    """Run the agentic loop until the model produces a final answer.

    Repeats: call model -> if it requests tools, execute them and feed
    results back -> otherwise stop. `max_turns` is a safety cap so a
    misbehaving agent can't loop forever.
    """
    messages = [{"role": "user", "content": question}]

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1000,
            tools=TOOLS,
            messages=messages,
        )

        # If the model is done (no tool requested), return its text answer
        if response.stop_reason != "tool_use":
            final_text = "".join(
                b.text for b in response.content if b.type == "text"
            )
            if verbose:
                print(f"[turn {turn}] model finished.\n")
            return final_text

        # Otherwise, record the model's turn and execute every tool it asked for
        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                if verbose:
                    print(f"[turn {turn}] running tool '{block.name}':")
                    print("   ", block.input.get("sql", block.input))
                try:
                    result = query_structured_data(block.input["sql"])
                    payload = json.dumps(result)
                except Exception as e:
                    payload = f"ERROR: {e}"   # feed errors back so the model can recover
                    if verbose:
                        print("    -> error:", e)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": payload,
                })

        # Feed all tool results back in one user turn
        messages.append({"role": "user", "content": tool_results})

    return "[stopped: reached max_turns]"

In [ ]:
answer = run_agent(
    "How many accounts are there in total?"
)
print("FINAL ANSWER:\n", answer)

[turn 0] running tool 'query_structured_data':
    SELECT COUNT(*) as total_accounts FROM accounts
[turn 1] model finished.

FINAL ANSWER:
 There are **150 accounts** in total.


In [ ]:
answer = run_agent(
    "Which region has the unhealthiest customers"
)
print("FINAL ANSWER:\n", answer)

[turn 0] running tool 'query_structured_data':
    SELECT region, COUNT(*) as unhealthy_count FROM accounts WHERE health_status = 'At Risk' GROUP BY region ORDER BY unhealthy_count DESC
[turn 1] model finished.

FINAL ANSWER:
 **AMER** has the unhealthiest customers with **22 accounts** at "At Risk" health status, followed by EMEA with 15 and APAC with 11 at-risk accounts.
